# ♻️ Garbage Classification — Training Demo

This notebook walks through training the garbage classifier defined in `src/` and shows the evaluation results. All the real logic (data loading, model, training loop) lives in `src/` as plain Python modules — this notebook just calls it and displays the results, which keeps the training code testable and reusable instead of trapped inside notebook cells.

**To run this on Google Colab:**
1. Runtime → Change runtime type → GPU (T4 is fine)
2. Run the cells top to bottom

## 1. Clone the repo and install dependencies

In [ ]:
# Clone the project repo and install dependencies
!git clone https://github.com/shahndaa/Garbage-Image-Classification.git
%cd Garbage-Image-Classification
!pip install -q -r requirements.txt

## 2. Get the dataset

This project uses the classic **TrashNet** dataset (Gary Thung & Mindy Yang, 2016): 2,527 images across 6 classes — cardboard, glass, metal, paper, plastic, trash.

Download it once and place it so that `data/dataset-resized/` directly contains the 6 class folders. The cell below does this automatically from the original source repo.

In [ ]:
import os
if not os.path.isdir('data/dataset-resized'):
    !git clone --depth 1 https://github.com/garythung/trashnet.git /tmp/trashnet
    !mkdir -p data
    !unzip -q /tmp/trashnet/data/dataset-resized.zip -d data/

print(sorted(os.listdir('data/dataset-resized')))

## 3. Train
Trains the classification head on frozen MobileNetV2, then fine-tunes the top layers. Takes ~10-15 minutes on a Colab T4 GPU.

In [ ]:
!python -m src.train

## 4. Look at the results

In [ ]:
from IPython.display import Image, display
display(Image('assets/training_curves.png'))
display(Image('assets/confusion_matrix.png'))

In [ ]:
import json
with open('assets/test_classification_report.json') as f:
    report = json.load(f)
print(f"Test accuracy: {report['accuracy']:.2%}")
print(f"Macro F1:      {report['macro avg']['f1-score']:.2%}")
for cls in ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']:
    m = report[cls]
    print(f"  {cls:10s} precision={m['precision']:.2f}  recall={m['recall']:.2f}  f1={m['f1-score']:.2f}")

## 5. Try a prediction

In [ ]:
from PIL import Image as PILImage
from src import predict

model = predict.load_trained_model()
img_path = 'data/dataset-resized/plastic/plastic1.jpg'  # swap for any image path
result = predict.predict(model, PILImage.open(img_path))
print(result)

## 6. Launch the interactive demo
Opens a shareable Gradio link you can drop straight into your portfolio / CV.

In [ ]:
!python app/app.py